In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/MLProject/dataset"
OUT_DIR = "/content/drive/MyDrive/MLProject/smiles"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install rdkit selfies adjusttext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 63.4 MB/s eta 0:00:00


In [ ]:
# %%
import os
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit import RDLogger
import selfies as sf
from rdkit.Chem import DataStructs

from collections import Counter

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

# ── 전역 상수 ──────────────────────────────────────────────
SEED      = 42
N_BITS_MG    = 1024
RADIUS    = 4
N_COND    = 3
LATENT_DIM     = 128
EMB_DIM        = 64
DEC_GRU_DIM    = 128
DEC_N_LAYERS   = 1
DROPOUT        = 0.1
BETA           = 0.3
LAMBDA_FP      = 1.0
LAMBDA_CE      = 0.55
KL_WARMUP      = 200
FREE_BITS_PER_DIM = 0.05
WORD_DROPOUT   = 0.05
MS_BATCH =  12
BATCH     = 600
MS_REPEAT = 1

EPOCHS        = 550
LR            = 5e-4
CKPT_INTERVAL = 25

COND_LABELS = ["GLP1", "GCGR", "GIP"]

MS_NAMES = [
    "GLP-1(7-36)", "GIP(1-42)", "Glucagon(1-29)",
    "Exenatide", "Lixisenatide", "Liraglutide", "Albiglutide",
    "Taspoglutide", "Dulaglutide", "Semaglutide",
    "Tirzepatide", "Retatrutide",
]

MS_COND_VEC = {
    "GLP-1(7-36)":    [1, 0, 0],
    "Exenatide":      [1, 0, 0],
    "Lixisenatide":   [1, 0, 0],
    "Liraglutide":    [1, 0, 0],
    "Albiglutide":    [1, 0, 0],
    "Taspoglutide":   [1, 0, 0],
    "Dulaglutide":    [1, 0, 0],
    "Semaglutide":    [1, 0, 0],
    "Tirzepatide":    [1, 0, 1],
    "Retatrutide":    [1, 1, 1],
    "Glucagon(1-29)": [0, 1, 0],
    "GIP(1-42)":      [0, 0, 1],
}

# vocab/tokenizer (데이터 로드 후 초기화)
VOCAB_SIZE = None
MAX_LEN   = None
PAD_IDX = SOS_IDX = EOS_IDX = UNK_IDX = None
PAD, SOS, EOS, UNK = "<PAD>", "<SOS>", "<EOS>", "<UNK>"

DATA_DIR = r"c:\colab_data"
OUT_DIR = r"c:\final"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)

# ── 유틸 함수 ──────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


def canonicalize(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None: return None
    mol = Chem.RemoveHs(mol)
    # 가장 큰 fragment만 취함 (salt 제거)
    frags = Chem.GetMolFrags(mol, asMols=True)
    mol = max(frags, key=lambda m: m.GetNumAtoms())
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)

def smiles_to_selfies_str(smi):
    try:
        if smi in (None, "", "nan", "None"): return None
        return sf.encoder(str(smi))
    except Exception:
        return None


def smiles_to_fp(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None: return None

    morgan = AllChem.GetMorganFingerprintAsBitVect(mol, radius=RADIUS, nBits=N_BITS_MG)
    fp = np.asarray(morgan, dtype=np.float32)
    return fp


def is_peptide(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return False
    return Descriptors.MolWt(mol) >= 500


bpe_merges = {}
bpe_c2i    = {}
bpe_i2c    = {}


def encode_smi(smi):
    sf_str = smiles_to_selfies_str(smi)
    if sf_str is None: sf_str = ""
    ids = apply_bpe(sf_str, bpe_merges, bpe_c2i)
    tokens = [SOS_IDX] + ids + [EOS_IDX]
    if len(tokens) > MAX_LEN:
        tokens = tokens[:MAX_LEN - 1] + [EOS_IDX]
    return tokens

def ids_to_smiles(ids):
    filtered = []
    for i in ids:
        if i == EOS_IDX:
            break
        if i in (PAD_IDX, SOS_IDX):
            continue
        if i == UNK_IDX: continue
        filtered.append(i)
    # BPE 토큰을 이어붙이면 SELFIES 복원
    sf_str = "".join(bpe_i2c.get(i, "") for i in filtered)
    try:
        smi = sf.decoder(sf_str)
    except Exception:
        return "", False
    mol = Chem.MolFromSmiles(smi)
    return smi, mol is not None

def get_cond_vec(row):
    name = row.get("name", "")
    if name in MS_COND_VEC: return MS_COND_VEC[name]
    target = str(row.get("target", "")).strip().upper()
    if target in ("GLP1R", "GLP-1R", "GLP1"): return [1, 0, 0]
    if target in ("GCGR",):                    return [0, 1, 0]
    if target in ("GIPR", "GIP"):              return [0, 0, 1]
    return [1, 0, 0]



def build_selfies_bpe(selfies_list, vocab_size=300):
    global VOCAB_SIZE, MAX_LEN, PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX

    # 초기 vocab: SELFIES 단일 토큰들
    base_vocab = sorted({
        tok
        for sf_str in selfies_list
        for tok in sf.split_selfies(sf_str)
    })

    # 각 SELFIES를 토큰 리스트로 변환
    corpus = [tuple(sf.split_selfies(s)) for s in selfies_list]

    merges = {}
    vocab  = set(base_vocab)

    n_merges = vocab_size - len(base_vocab) - 4  # special tokens 제외
    n_merges = max(0, n_merges)

    for _ in range(n_merges):
        # 인접 토큰 쌍 빈도 계산
        pair_freq = Counter()
        for seq in corpus:
            for i in range(len(seq) - 1):
                pair_freq[(seq[i], seq[i+1])] += 1
        if not pair_freq:
            break
        best = max(pair_freq, key=pair_freq.get)
        if pair_freq[best] < 2:
            break

        # 병합
        merged = best[0] + best[1]
        merges[best] = merged
        vocab.add(merged)

        new_corpus = []
        for seq in corpus:
            new_seq = []
            i = 0
            while i < len(seq):
                if i < len(seq) - 1 and (seq[i], seq[i+1]) == best:
                    new_seq.append(merged)
                    i += 2
                else:
                    new_seq.append(seq[i])
                    i += 1
            new_corpus.append(tuple(new_seq))
        corpus = new_corpus

    # 전역변수 설정
    special = [PAD, SOS, EOS, UNK]
    full_vocab = special + sorted(vocab)
    c2i = {c: i for i, c in enumerate(full_vocab)}
    i2c = {i: c for c, i in c2i.items()}

    PAD_IDX = c2i[PAD]; SOS_IDX = c2i[SOS]
    EOS_IDX = c2i[EOS]; UNK_IDX = c2i[UNK]
    VOCAB_SIZE = len(full_vocab)

    # MAX_LEN 계산을 corpus 기준이 아니라 원본 selfies 기준으로
    lens   = [len(apply_bpe(s, merges, c2i)) + 2 for s in selfies_list]
    MAX_LEN = max(lens) + 5  # 여유 추가

    print(f"BPE Vocab: {VOCAB_SIZE}  MAX_LEN: {MAX_LEN}  Avg: {sum(lens)/len(lens):.1f}")
    return c2i, i2c, merges


def apply_bpe(sf_str_or_toks, merges, c2i):
    if isinstance(sf_str_or_toks, (list, tuple)):
        toks = tuple(sf_str_or_toks)
    else:
        toks = tuple(sf.split_selfies(sf_str_or_toks)) if sf_str_or_toks else ()
    for (a, b), merged in merges.items():
        new_toks = []
        i = 0
        while i < len(toks):
            if i < len(toks)-1 and toks[i]==a and toks[i+1]==b:
                new_toks.append(merged)
                i += 2
            else:
                new_toks.append(toks[i])
                i += 1
        toks = tuple(new_toks)
    return [c2i.get(t, UNK_IDX) for t in toks]


# ── Dataset ────────────────────────────────────────────────
class SMILESFPDataset(Dataset):
    def __init__(self, frame):
        self.samples    = []
        self.valid_rows = []
        for _, row in frame.iterrows():
            smi = str(row["smiles"])
            if smi in ("nan", "", "None"): continue
            fp = smiles_to_fp(smi)
            if fp is None: continue
            tokens = encode_smi(smi)
            padded = tokens + [PAD_IDX] * (MAX_LEN - len(tokens))
            cond   = torch.tensor(get_cond_vec(row.to_dict()), dtype=torch.float32)
            self.samples.append({
                "fp":         torch.tensor(fp,           dtype=torch.float32),
                "tokens":     torch.tensor(padded,       dtype=torch.long),
                "tokens_in":  torch.tensor(padded[:-1],  dtype=torch.long),
                "tokens_out": torch.tensor(padded[1:],   dtype=torch.long),
                "cond":       cond,
            })
            self.valid_rows.append(row.to_dict())

    def __len__(self):        return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate_batch(batch):
    return {
        "fp":         torch.stack([d["fp"]         for d in batch]),
        "tokens":     torch.stack([d["tokens"]     for d in batch]),
        "tokens_in":  torch.stack([d["tokens_in"]  for d in batch]),
        "tokens_out": torch.stack([d["tokens_out"] for d in batch]),
        "cond":       torch.stack([d["cond"]       for d in batch]),
    }

class MilestoneLoader:
    def __init__(self, ds_bg, batch_size, ms_repeat, ms_batch_size, ds_ms, seed=SEED):
        self.ds_bg = ds_bg
        self.batch_size = batch_size
        self.ms_batch_size = ms_batch_size
        self.ms_repeat = ms_repeat
        self.ds_ms = ds_ms
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.ds_bg) // self.batch_size + self.ms_repeat

    def __iter__(self):
        perm = self.rng.permutation(len(self.ds_bg))
        n_bg_bat = len(self.ds_bg) // self.batch_size

        ms_slots = set(
            int(n_bg_bat * (i + 1) / (self.ms_repeat + 1))
            for i in range(self.ms_repeat)
        )

        ms_count = 0
        for b in range(n_bg_bat):
            if b in ms_slots:
                yield self.sample_balanced_ms_batch()
                ms_count += 1

            idxs = perm[b * self.batch_size:(b + 1) * self.batch_size]
            yield collate_batch([self.ds_bg[int(i)] for i in idxs])

        while ms_count < self.ms_repeat:
            yield self.sample_balanced_ms_batch()
            ms_count += 1

    def sample_balanced_ms_batch(self):
        n_ms = len(self.ds_ms)
        if n_ms == 0:
            raise ValueError("No milestone samples found. Check MS_NAMES / df_ms.")

        per = self.ms_batch_size // n_ms
        rem = self.ms_batch_size % n_ms

        idxs = []
        for m in range(n_ms):
            idxs.extend([m] * per)

        for _ in range(rem):
            idxs.append(int(self.rng.integers(0, n_ms)))

        self.rng.shuffle(idxs)
        return collate_batch([self.ds_ms[int(i)] for i in idxs])


# ── Model ──────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        # FP branch
        self.fp_net = nn.Sequential(
            nn.Linear(N_BITS_MG, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

        self.seq_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM, padding_idx=PAD_IDX)

        self.ch = 64
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(EMB_DIM, self.ch, kernel_size=k, padding=k // 2),
                nn.GELU(),
                nn.Conv1d(self.ch, self.ch, kernel_size=k, padding=k // 2),
                nn.GELU(),
            )
            for k in [3, 4, 5]
        ])
        CNN_OUT = self.ch * 3

        self.fuse = nn.Sequential(
            nn.Linear(256 + CNN_OUT + N_COND, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

        self.fc_mu     = nn.Linear(256, LATENT_DIM)
        self.fc_logvar = nn.Linear(256, LATENT_DIM)

    def forward(self, fp, cond, tokens=None):
        fp_h = self.fp_net(fp)

        if tokens is not None:
            emb = self.seq_emb(tokens)
            x = emb.permute(0, 2, 1)

            pooled = []
            for conv in self.convs:
                c = conv(x)
                pooled.append(c.max(dim=-1).values)

            seq_h = torch.cat(pooled, dim=-1)
        else:
            seq_h = torch.zeros(fp.size(0), self.ch * len(self.convs), device=fp.device)

        h = self.fuse(torch.cat([fp_h, seq_h, cond], dim=-1))
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM + N_COND, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, 512),                  nn.LayerNorm(512), nn.GELU(),
            nn.Linear(512, N_BITS_MG),               nn.Sigmoid(),
        )

    def forward(self, z, cond):
        return self.net(torch.cat([z, cond], dim=-1))


class GRUSELFIESDecoder(nn.Module):
    """매 스텝 emb + z + cond concat (Option C)"""
    def __init__(self):
        super().__init__()
        gru_in = EMB_DIM + LATENT_DIM + N_COND
        self.fc_init   = nn.Linear(LATENT_DIM + N_COND, DEC_GRU_DIM * DEC_N_LAYERS)
        self.token_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM, padding_idx=PAD_IDX)
        self.gru       = nn.GRU(gru_in, DEC_GRU_DIM,
                                num_layers=DEC_N_LAYERS, batch_first=True)
        self.dropout   = nn.Dropout(DROPOUT)
        self.fc_out    = nn.Linear(DEC_GRU_DIM, VOCAB_SIZE)

    def _init_hidden(self, z, cond):
        h = torch.tanh(self.fc_init(torch.cat([z, cond], dim=-1)))
        B = z.size(0)
        return h.view(B, DEC_N_LAYERS, DEC_GRU_DIM).permute(1, 0, 2).contiguous()

    def forward(self, z, cond, tokens_in, word_dropout=0.0):
        B, L = tokens_in.shape
        h0   = self._init_hidden(z, cond)
        tok  = tokens_in.clone()
        if word_dropout > 0.0 and self.training:
            mask = (torch.rand(B, L, device=tok.device) < word_dropout)
            mask &= (tok != PAD_IDX) & (tok != SOS_IDX) & (tok != EOS_IDX)
            tok[mask] = UNK_IDX
        emb      = self.dropout(self.token_emb(tok))
        z_exp    = z.unsqueeze(1).expand(B, L, -1)
        cond_exp = cond.unsqueeze(1).expand(B, L, -1)
        inp      = torch.cat([emb, z_exp, cond_exp], dim=-1)
        out, _   = self.gru(inp, h0)
        return self.fc_out(self.dropout(out))

    @torch.no_grad()
    def greedy(self, z, cond, max_len=None):
        if max_len is None:
            max_len = MAX_LEN

        B        = z.size(0)
        h        = self._init_hidden(z, cond)
        token    = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=z.device)
        gen_ids  = [[] for _ in range(B)]
        finished = torch.zeros(B, dtype=torch.bool, device=z.device)
        for _ in range(max_len):
            emb      = self.token_emb(token)
            z_exp    = z.unsqueeze(1)
            cond_exp = cond.unsqueeze(1)
            inp      = torch.cat([emb, z_exp, cond_exp], dim=-1)
            out, h   = self.gru(inp, h)
            logit    = self.fc_out(out.squeeze(1))
            logit[:, UNK_IDX] = -1e9
            next_tok = logit.argmax(dim=-1)
            for b in range(B):
                if not finished[b]:
                    t = next_tok[b].item()
                    gen_ids[b].append(t)
                    if t == EOS_IDX: finished[b] = True
            if finished.all(): break
            # finished된 beam은 PAD 입력으로 대체 (불필요한 GRU 연산 방지)
            next_tok = torch.where(finished, torch.full_like(next_tok, PAD_IDX), next_tok)
            token = next_tok.unsqueeze(1)
        return gen_ids[0] if B == 1 else gen_ids


    @torch.no_grad()
    def beam_search(self, z, cond, beam_width=5, max_len=None):
        if max_len is None:
            max_len = MAX_LEN

        assert z.size(0) == 1, "beam_search only supports batch size 1"
        if max_len is None: max_len = MAX_LEN
        device = z.device
        h = self._init_hidden(z, cond)
        beams = [(0.0, [SOS_IDX], h)]
        completed = []

        for _ in range(max_len):
            new_beams = []
            for score, ids, h_prev in beams:
                if ids[-1] == EOS_IDX:
                    completed.append((score, ids))
                    continue
                token    = torch.tensor([[ids[-1]]], dtype=torch.long, device=device)
                emb      = self.token_emb(token)
                z_exp    = z.unsqueeze(1)
                cond_exp = cond.unsqueeze(1)
                inp      = torch.cat([emb, z_exp, cond_exp], dim=-1)
                out, h_new = self.gru(inp, h_prev)
                logit    = self.fc_out(out.squeeze(1))
                logit[0, UNK_IDX] = -1e9
                log_probs = torch.log_softmax(logit, dim=-1).squeeze(0)
                topk_scores, topk_ids = log_probs.topk(beam_width)
                for tok_score, tok_id in zip(topk_scores.tolist(), topk_ids.tolist()):
                    # h_new를 clone해서 beam간 참조 공유 방지
                    new_beams.append((score + tok_score, ids + [tok_id], h_new.clone()))

            if not new_beams: break
            new_beams.sort(key=lambda x: x[0], reverse=True)
            beams = new_beams[:beam_width]
            if all(b[1][-1] == EOS_IDX for b in beams):
                for b in beams: completed.append((b[0], b[1]))
                break

        if not completed:
            completed = [(b[0], b[1]) for b in beams]

        best_score, best_ids = max(completed, key=lambda x: x[0] / max(len(x[1]), 1))
        result = []
        for tid in best_ids:
            if tid == SOS_IDX: continue
            if tid == EOS_IDX: break
            result.append(tid)
        return result


    @torch.no_grad()
    def sample(self, z, cond, max_len=None, temperature=0.9, top_k=0, top_p=0.9,
        repetition_penalty=1.1, min_len=5, eos_boost=0.0):
        if max_len is None:
            max_len = MAX_LEN

        B = z.size(0)
        device = z.device
        h = self._init_hidden(z, cond)
        token = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=device)

        gen_ids  = [[] for _ in range(B)]
        finished = torch.zeros(B, dtype=torch.bool, device=device)

        for step in range(max_len):
            emb = self.token_emb(token)
            z_exp    = z.unsqueeze(1)
            cond_exp = cond.unsqueeze(1)
            inp = torch.cat([emb, z_exp, cond_exp], dim=-1)
            out, h = self.gru(inp, h)
            logits = self.fc_out(out.squeeze(1))

            logits[:, UNK_IDX] = -1e9
            logits[:, PAD_IDX] = -1e9
            if step < min_len:
                logits[:, EOS_IDX] = -1e9
            else:
                logits[:, EOS_IDX] += eos_boost * (step - min_len)

            # temperature 먼저 적용
            logits = logits / max(temperature, 1e-6)

            # penalty는 temperature 이후 raw scale에서 적용
            if repetition_penalty and repetition_penalty > 1.0:
                for b in range(B):
                    if finished[b]:
                        continue
                    for tid in set(gen_ids[b][-8:]):
                        val = logits[b, tid]
                        if val > 0:
                            logits[b, tid] = val / repetition_penalty
                        else:
                            logits[b, tid] = val * repetition_penalty

            if top_k and top_k > 0:
                k = min(top_k, logits.size(-1))
                kth = torch.topk(logits, k, dim=-1).values[:, -1].unsqueeze(-1)
                logits = torch.where(logits < kth, torch.full_like(logits, -1e9), logits)

            if top_p and top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                sorted_probs  = torch.softmax(sorted_logits, dim=-1)
                cum_probs     = torch.cumsum(sorted_probs, dim=-1)
                remove        = cum_probs > top_p
                remove[:, 1:] = remove[:, :-1].clone()
                remove[:, 0]  = False
                sorted_logits = sorted_logits.masked_fill(remove, -1e9)
                logits        = torch.full_like(logits, -1e9)
                logits.scatter_(1, sorted_idx, sorted_logits)

            probs    = torch.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(1)

            for b in range(B):
                if finished[b]:
                    continue
                t = next_tok[b].item()
                gen_ids[b].append(t)
                if t == EOS_IDX:
                    finished[b] = True

            if finished.all():
                break

            next_tok = torch.where(
                finished,
                torch.full_like(next_tok, PAD_IDX),  # EOS 대신 PAD 입력
                next_tok
            )
            token = next_tok.unsqueeze(1)

        cleaned = []
        for ids in gen_ids:
            seq = []
            for tid in ids:
                if tid == EOS_IDX: break
                if tid not in (PAD_IDX, SOS_IDX): seq.append(tid)
            cleaned.append(seq)

        return cleaned[0] if B == 1 else cleaned

class FPGRUcVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder     = Encoder()
        self.fp_decoder  = Decoder()
        self.smi_decoder = GRUSELFIESDecoder()


    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + std * torch.randn_like(std)
        return mu

    def encode(self, fp, cond, tokens=None):
        return self.encoder(fp, cond, tokens=tokens)

    def forward(self, fp, cond, tokens_in, tokens=None, word_dropout=0.0):
        mu, logvar = self.encoder(fp, cond, tokens=tokens)
        z = self.reparameterize(mu, logvar)
        fp_recon = self.fp_decoder(z, cond)
        smi_logits = self.smi_decoder(z, cond, tokens_in, word_dropout)
        return fp_recon, smi_logits, mu, logvar

# ── Loss ───────────────────────────────────────────────────
def cvae_loss(fp_recon, fp_target, smi_logits, smi_target, mu, logvar, beta,
              free_bits=FREE_BITS_PER_DIM):
    loss_fp = F.binary_cross_entropy(fp_recon, fp_target, reduction="mean")

    B, L, V = smi_logits.shape
    n_valid = (smi_target != PAD_IDX).sum().item() + 1e-8
    loss_ce = F.cross_entropy(
        smi_logits.reshape(B * L, V),
        smi_target.reshape(B * L),
        ignore_index=PAD_IDX,
        reduction="sum"
    ) / n_valid
    kl_dim = -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp())
    kl_dim_mean = kl_dim.mean(dim=0)
    kl_free = torch.clamp(kl_dim_mean, min=free_bits).sum()

    total = LAMBDA_FP * loss_fp + LAMBDA_CE * loss_ce + beta * kl_free
    return total, loss_fp.item(), loss_ce.item(), kl_dim_mean.sum().item(), kl_free.item()

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


if __name__ == "__main__":
    set_seed(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {DEVICE}")

    # ── 데이터 로드 ────────────────────────────────────────
    df = pd.read_csv(f"{DATA_DIR}/SMILES.csv")

    df["smiles"] = df["smiles"].apply(canonicalize)
    df = df[df["smiles"].notna()].reset_index(drop=True)

    before = len(df)
    df = df[df["smiles"].apply(is_peptide)].reset_index(drop=True)
    print(f"필터 (MW>=500): {before}건 → {len(df)}건")

    df["selfies"] = df["smiles"].apply(smiles_to_selfies_str)
    df = df[df["selfies"].notna()].reset_index(drop=True)

    is_ms  = df["name"].isin(MS_NAMES)
    df_ms  = df[is_ms].reset_index(drop=True)
    df_bg  = df[~is_ms].reset_index(drop=True)

    all_selfies = df["selfies"].tolist()
    _c2i, _i2c, _merges = build_selfies_bpe(all_selfies, vocab_size=250)

    bpe_c2i.clear()
    bpe_i2c.clear()
    bpe_merges.clear()

    bpe_c2i.update(_c2i)
    bpe_i2c.update(_i2c)
    bpe_merges.update(_merges)

    print(f"Vocab: {VOCAB_SIZE}  MAX_LEN: {MAX_LEN}")

    # ── Dataset / Loader ───────────────────────────────────
    ds_bg = SMILESFPDataset(df_bg)
    ds_ms = SMILESFPDataset(df_ms)

    N_MS      = len(ds_ms)
    ms_names  = df_ms["name"].tolist()
    ms_idx    = {n: i for i, n in enumerate(ms_names)}
    ms_fps    = torch.stack([ds_ms[i]["fp"]     for i in range(N_MS)]).to(DEVICE)
    ms_tokens = torch.stack([ds_ms[i]["tokens"] for i in range(N_MS)]).to(DEVICE)
    ms_cond   = torch.stack([ds_ms[i]["cond"]   for i in range(N_MS)]).to(DEVICE)

    loader = MilestoneLoader(ds_bg, BATCH, MS_REPEAT, MS_BATCH, ds_ms, seed=SEED)
    print(f"Dataset: bg={len(ds_bg)}, ms={N_MS}")

    # ── 모델 ───────────────────────────────────────────────
    model  = FPGRUcVAE().to(DEVICE)
    params = sum(p.numel() for p in model.parameters())
    print(f"\nParameters: {params:,}")

    # ── 학습 ───────────────────────────────────────────────
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    def eval_direction():
        model.eval()
        with torch.no_grad():
            mu, _ = model.encode(ms_fps, ms_cond, tokens=ms_tokens)
        Z = mu.cpu().numpy()
        needed = ["GLP-1(7-36)", "Exenatide", "Liraglutide", "Semaglutide", "Taspoglutide"]
        if not all(n in ms_idx for n in needed): return -999.0, -999.0
        def gz(n): return Z[ms_idx[n]]
        v_evo  = gz("Semaglutide")  - gz("GLP-1(7-36)")
        v_adv = gz("Tirzepatide")  - gz("Semaglutide")
        return cosine(v_evo, v_adv)


    def eval_reconstruction_tani():
        model.eval()
        tanis = []

        with torch.no_grad():
            mu, _ = model.encode(ms_fps, ms_cond, tokens=ms_tokens)

            for b in range(mu.size(0)):
                orig_smi = df_ms.loc[ms_idx[ms_names[b]], "smiles"]
                ids = model.smi_decoder.greedy(mu[b:b+1], ms_cond[b:b+1])

                recon_smi, valid = ids_to_smiles(ids)
                if not valid:
                    continue

                mol_o = Chem.MolFromSmiles(orig_smi)
                mol_r = Chem.MolFromSmiles(recon_smi)
                if mol_o is None or mol_r is None:
                    continue


                fp_o = AllChem.GetMorganFingerprintAsBitVect(
                    mol_o, radius=RADIUS, nBits=N_BITS_MG
                )
                fp_r = AllChem.GetMorganFingerprintAsBitVect(
                    mol_r, radius=RADIUS, nBits=N_BITS_MG
                )
                tanis.append(DataStructs.TanimotoSimilarity(fp_o, fp_r))

        return float(np.mean(tanis)) if tanis else 0.0

    history = {"loss": [], "fp": [], "ce": [], "kl_raw": [], "kl_free": [], "evo_d": []}
    checkpoints   = {}
    best_evo_d    = -999.0
    best_evo_d_ep = -1
    best_state    = None

    print(f"\n{'Ep':>5} {'Loss':>9} {'FP':>8} {'CE':>8} {'KLraw':>8} {'KLfree':>8} {'Evo↔D':>8} {'Tani':>7}")
    print("-" * 82)

    eval_records = []

    for ep in range(1, EPOCHS + 1):
        beta_now = BETA * min(1.0, ep / KL_WARMUP)
        model.train()
        ep_loss = ep_fp = ep_ce = ep_kl_raw = ep_kl_free = 0.0
        n_batches = 0

        for batch in loader:
            fp      = batch["fp"].to(DEVICE)
            tok_in  = batch["tokens_in"].to(DEVICE)
            tok_out = batch["tokens_out"].to(DEVICE)
            cond    = batch["cond"].to(DEVICE)
            tokens  = batch["tokens"].to(DEVICE)

            optimizer.zero_grad()

            fp_recon, smi_logits, mu, logvar = model(
                fp, cond, tok_in,
                tokens=tokens,
                word_dropout=WORD_DROPOUT
            )

            loss, fp_l, ce_l, kl_raw, kl_free = cvae_loss(
                fp_recon, fp, smi_logits, tok_out, mu, logvar, beta_now
            )

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            ep_loss += loss.item()
            ep_fp += fp_l
            ep_ce += ce_l
            ep_kl_raw += kl_raw
            ep_kl_free += kl_free
            n_batches += 1

        avg = lambda x: x / n_batches
        history["loss"].append(avg(ep_loss))
        history["fp"].append(avg(ep_fp))
        history["ce"].append(avg(ep_ce))
        history["kl_raw"].append(avg(ep_kl_raw))
        history["kl_free"].append(avg(ep_kl_free))

        if ep % CKPT_INTERVAL == 0 or ep == EPOCHS:
            state_cpu = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            checkpoints[ep] = state_cpu
            torch.save(
                {"epoch": ep, "state_dict": state_cpu, "metadata": None},
                f"{OUT_DIR}/checkpoints/cvae_gru_ep{ep:04d}.pt"
            )

        if ep % CKPT_INTERVAL == 0 or ep == 1:
            ce_d = eval_direction()
            tani = eval_reconstruction_tani()

            history["evo_d"].append(ce_d)

            if ce_d > best_evo_d:
                best_evo_d    = ce_d
                best_evo_d_ep = ep
                best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            print(
                f"{ep:>5} {avg(ep_loss):>9.4f} {avg(ep_fp):>8.4f} {avg(ep_ce):>8.4f} "
                f"{avg(ep_kl_raw):>8.4f} {avg(ep_kl_free):>8.4f} "
                f"{ce_d:>+8.4f} {tani:>7.3f}"
            )

            eval_records.append({
                "epoch": ep,
                "evo_d": ce_d,
                "tani": tani
            })

    last_ep = max(checkpoints.keys())

    hist_df = pd.DataFrame({
        "epoch": np.arange(1, EPOCHS + 1),
        "loss": history["loss"],
        "fp": history["fp"],
        "ce": history["ce"],
        "kl_raw": history["kl_raw"],
        "kl_free": history["kl_free"],
    })
    hist_df.to_csv(f"{OUT_DIR}/train_history.csv", index=False)

    pd.DataFrame(eval_records).to_csv(f"{OUT_DIR}/eval_history.csv", index=False)
    metadata = {
        "seed": SEED,
        "fingerprint": {"n_bits_mg": N_BITS_MG, "radius": RADIUS},
        "condition": {
            "cond_labels": COND_LABELS, "n_cond": N_COND,
            "ms_cond_vec": MS_COND_VEC,
        },
        "model": {
            "emb_dim": EMB_DIM, "latent_dim": LATENT_DIM,
            "dec_gru_dim": DEC_GRU_DIM, "dec_n_layers": DEC_N_LAYERS,
            "dropout": DROPOUT,
            "encoder": "fp_plus_selfies_cnn",
        },
        "loss": {
            "beta": BETA, "lambda_fp": LAMBDA_FP, "lambda_ce": LAMBDA_CE,
            "free_bits": FREE_BITS_PER_DIM, "word_dropout": WORD_DROPOUT,
        },
        "training": {
            "epochs": EPOCHS,
            "lr": LR,
            "batch": BATCH,
            "ms_repeat": MS_REPEAT,
            "ms_batch": MS_BATCH,
            "ckpt_interval": CKPT_INTERVAL,
        },
        "tokenizer": {
            "type": "bpe",
            "vocab_size": VOCAB_SIZE,
            "max_len": MAX_LEN,
            "pad_idx": PAD_IDX, "sos_idx": SOS_IDX,
            "eos_idx": EOS_IDX, "unk_idx": UNK_IDX,
            "bpe_c2i": bpe_c2i,
            "bpe_i2c": {str(k): v for k, v in bpe_i2c.items()},
            "bpe_merges": {f"{k[0]}\x00{k[1]}": v for k, v in bpe_merges.items()},
        },
    }


    for ep_ck, state in sorted(checkpoints.items()):
        torch.save(
            {"epoch": ep_ck, "state_dict": state, "metadata": metadata},
            f"{OUT_DIR}/checkpoints/cvae_gru_ep{ep_ck:04d}.pt"
        )

    torch.save({"epoch": last_ep, "state_dict": checkpoints[last_ep], "metadata": metadata},
               f"{OUT_DIR}/cvae_gru_last.pt")
    torch.save({"epoch": best_evo_d_ep, "state_dict": best_state, "metadata": metadata},
               f"{OUT_DIR}/cvae_gru_best_evod.pt")
    print(f"\n→ last      : ep {last_ep}")
    print(f"→ best Evo↔D: ep {best_evo_d_ep}  ({best_evo_d:+.4f})")

    # checkpoint 요약
    print("\n" + "─"*55)
    print(f"{'Epoch':>7} {'Evo↔Dual':>10} verdict")
    print("-"*48)
    for ep_ck, state in sorted(checkpoints.items()):
        model.load_state_dict(state)
        ce_d = eval_direction()
        print(f"{ep_ck:>7} {ce_d:>+10.4f} {'✅' if ce_d>0 else '❌'}")

    model.load_state_dict(checkpoints[last_ep])


Device: cuda
필터 (MW>=500): 3540건 → 1874건
BPE Vocab: 250  MAX_LEN: 116  Avg: 41.7
Vocab: 250  MAX_LEN: 116
Dataset: bg=1862, ms=12

Parameters: 1,836,538

   Ep      Loss       FP       CE    KLraw   KLfree    Evo↔D    Tani
----------------------------------------------------------------------------------
    1    3.7247   0.6740   5.5057  14.9689  15.0789  -0.1319   0.052
   25    2.9826   0.3106   4.4076   5.5563   6.6070  +0.1698   0.043
   50    2.7164   0.2844   3.5284   5.3495   6.5522  +0.3919   0.378
   75    2.5448   0.2556   2.8464   4.9913   6.4327  +0.4484   0.475
  100    2.5662   0.2568   2.4456   4.8221   6.4292  -0.1196   0.478
  125    2.6217   0.2344   2.1484   5.0709   6.4302  +0.2999   0.695
  150    2.7371   0.2266   1.9380   4.9964   6.4204  +0.2074   0.749
  175    2.8886   0.2241   1.7838   4.7581   6.4131  +0.3021   0.734
  200    3.0467   0.2204   1.6459   4.6861   6.4038  +0.4222   0.769
  225    3.0200   0.2293   1.5778   4.8039   6.4097  +0.2542   0.764
  25

# Evaluation Codes

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import torch

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, Descriptors
from rdkit import RDLogger
import selfies as sf

import umap
import seaborn as sns
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

try:
    from adjustText import adjust_text
    HAS_ADJ = True
except ImportError:
    HAS_ADJ = False

# Colab / Drive 경로
DATA_DIR = "/content/drive/MyDrive/MLProject/dataset"
OUT_DIR  = "/content/drive/MyDrive/MLProject/smiles"

cp = 500
CKPT = f"{OUT_DIR}/checkpoints/cvae_gru_ep{cp:04d}.pt"

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CKPT}")
print("exists?:", os.path.exists(CKPT))

Device: cuda
Checkpoint: /content/drive/MyDrive/MLProject/smiles/checkpoints/cvae_gru_ep0500.pt
exists?: True


In [ ]:
# ══════════════════════════════════════════════════════════
#  Checkpoint / Metadata / Dataset 로드
# ══════════════════════════════════════════════════════════

# ── 체크포인트 로드 ────────────────────────────────────────
ckpt       = torch.load(CKPT, map_location="cpu", weights_only=False)
meta       = ckpt["metadata"]
state_dict = ckpt["state_dict"]

fp_meta    = meta.get("fingerprint", {})
cond_meta  = meta.get("condition", {})
model_meta = meta.get("model", {})
loss_meta  = meta.get("loss", {})

# ── metadata → 전역 하이퍼파라미터 복원 ─────────────────────
N_BITS_MG    = fp_meta.get("n_bits_mg", N_BITS_MG)
RADIUS       = fp_meta.get("radius", RADIUS)
N_COND       = cond_meta.get("n_cond", N_COND)

LATENT_DIM   = model_meta.get("latent_dim", LATENT_DIM)
EMB_DIM      = model_meta.get("emb_dim", EMB_DIM)
DEC_GRU_DIM  = model_meta.get("dec_gru_dim", DEC_GRU_DIM)
DEC_N_LAYERS = model_meta.get("dec_n_layers", DEC_N_LAYERS)
DROPOUT      = model_meta.get("dropout", DROPOUT)

LAMBDA_FP = loss_meta.get("lambda_fp", LAMBDA_FP)
LAMBDA_CE = loss_meta.get("lambda_ce", LAMBDA_CE)
FREE_BITS_PER_DIM = loss_meta.get("free_bits", FREE_BITS_PER_DIM)

# 조건/이름 metadata도 있으면 최신값으로 복원
COND_LABELS = cond_meta.get("cond_labels", COND_LABELS)
MS_COND_VEC = cond_meta.get("ms_cond_vec", MS_COND_VEC)


# ── vocab/tokenizer 복원 ──────────────────────────────────
tok = meta["tokenizer"]

bpe_c2i.clear()
bpe_i2c.clear()
bpe_merges.clear()

bpe_c2i.update(tok["bpe_c2i"])
bpe_i2c.update({int(k): v for k, v in tok["bpe_i2c"].items()})
bpe_merges.update({tuple(k.split("\x00")): v for k, v in tok["bpe_merges"].items()})

VOCAB_SIZE = tok["vocab_size"]
MAX_LEN    = tok["max_len"]

PAD_IDX = tok["pad_idx"]
SOS_IDX = tok["sos_idx"]
EOS_IDX = tok["eos_idx"]
UNK_IDX = tok["unk_idx"]

print(f"Vocab: {VOCAB_SIZE}  MAX_LEN: {MAX_LEN}  (from metadata)")
print(
    f"Model: latent={LATENT_DIM}, emb={EMB_DIM}, "
    f"gru={DEC_GRU_DIM}, layers={DEC_N_LAYERS}"
)
print(f"Encoder: {model_meta.get('encoder', 'N/A')}")


# ── 데이터 로드 ────────────────────────────────────────────
df = pd.read_csv(f"{DATA_DIR}/SMILES.csv")

# # 기존 evaluation 코드의 1차 길이 필터 유지
# df = df[df["smiles"].apply(lambda s: isinstance(s, str) and len(s) >= 40)].reset_index(drop=True)

df["smiles"] = df["smiles"].apply(canonicalize)
df = df[df["smiles"].notna()].reset_index(drop=True)

df = df[df["smiles"].apply(is_peptide)].reset_index(drop=True)

df["selfies"] = df["smiles"].apply(smiles_to_selfies_str)
df = df[df["selfies"].notna()].reset_index(drop=True)

is_ms = df["name"].isin(MS_NAMES)
df_ms = df[is_ms].reset_index(drop=True)
df_bg = df[~is_ms].reset_index(drop=True)

print(f"Background: {len(df_bg)}  Milestone: {len(df_ms)}")


# ── Dataset 생성 ──────────────────────────────────────────
ds_bg = SMILESFPDataset(df_bg)
ds_ms = SMILESFPDataset(df_ms)

ms_names = df_ms["name"].tolist()
ms_idx   = {n: i for i, n in enumerate(ms_names)}
N_MS     = len(ds_ms)

ms_fps    = torch.stack([ds_ms[i]["fp"]     for i in range(N_MS)]).to(DEVICE)
ms_tokens = torch.stack([ds_ms[i]["tokens"] for i in range(N_MS)]).to(DEVICE)
ms_cond   = torch.stack([ds_ms[i]["cond"]   for i in range(N_MS)]).to(DEVICE)

df_bg_valid = pd.DataFrame(ds_bg.valid_rows).reset_index(drop=True)

y_camp = (
    df_bg_valid["cAMP"].values.astype(np.float32)
    if "cAMP" in df_bg_valid.columns
    else np.full(len(ds_bg), np.nan)
)

y_bias = (
    df_bg_valid["bias_ratio_cAMP_minus_barr"].values.astype(np.float32)
    if "bias_ratio_cAMP_minus_barr" in df_bg_valid.columns
    else np.full(len(ds_bg), np.nan)
)


# ══════════════════════════════════════════════════════════
#  모델 로드
# ══════════════════════════════════════════════════════════
model = FPGRUcVAE().to(DEVICE)

# checkpoint가 CPU 저장본이면 그대로 로드 가능
model.load_state_dict(state_dict)

model.eval()

params = sum(p.numel() for p in model.parameters())
print(f"✅ 모델 로드 완료  Parameters: {params:,}")

Vocab: 250  MAX_LEN: 116  (from metadata)
Model: latent=128, emb=64, gru=128, layers=1
Encoder: fp_plus_selfies_cnn
Background: 1862  Milestone: 12
✅ 모델 로드 완료  Parameters: 1,836,538


In [ ]:
# ══════════════════════════════════════════════════════════
#  유틸 함수
# ══════════════════════════════════════════════════════════

def normalize_smiles(smi):
    try:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            return ""
        return Chem.MolToSmiles(Chem.RemoveHs(mol), canonical=True, isomericSmiles=False)
    except Exception:
        return ""

def normalize_via_selfies(smi):
    """SMILES -> SELFIES -> SMILES로 한 번 통과시켜 evaluation canonical 기준을 맞춤."""
    try:
        sf_str = sf.encoder(str(smi))
        dec_smi = sf.decoder(sf_str)
        mol = Chem.MolFromSmiles(dec_smi)
        if mol is None:
            return ""
        return Chem.MolToSmiles(Chem.RemoveHs(mol), canonical=True, isomericSmiles=False)
    except Exception:
        return ""

def decode_tokens(token_ids):
    toks = []
    for tid in token_ids:
        tid = int(tid)
        if tid == EOS_IDX:
            break
        if tid in (SOS_IDX, PAD_IDX, UNK_IDX):
            continue

        tok = bpe_i2c.get(tid, "")
        if tok:
            toks.append(tok)

    sf_str = "".join(toks)
    try:
        return sf.decoder(sf_str) if sf_str else ""
    except Exception:
        return ""

def check_valid(smi):
    return bool(smi) and Chem.MolFromSmiles(smi) is not None

def fp_tanimoto(smi_a, smi_b):
    try:
        ma = Chem.MolFromSmiles(str(smi_a))
        mb = Chem.MolFromSmiles(str(smi_b))
        if ma is None or mb is None:
            return float("nan")
        fa = AllChem.GetMorganFingerprintAsBitVect(ma, radius=RADIUS, nBits=N_BITS_MG)
        fb = AllChem.GetMorganFingerprintAsBitVect(mb, radius=RADIUS, nBits=N_BITS_MG)
        return DataStructs.TanimotoSimilarity(fa, fb)
    except Exception:
        return float("nan")

def _mol(smi):
    try:
        return Chem.MolFromSmiles(str(smi))
    except Exception:
        return None

def get_mw(smi):
    mol = _mol(smi)
    return Descriptors.MolWt(mol) if mol else np.nan

def get_logp(smi):
    mol = _mol(smi)
    return Descriptors.MolLogP(mol) if mol else np.nan

def get_tpsa(smi):
    mol = _mol(smi)
    return Descriptors.TPSA(mol) if mol else np.nan

def get_peptide_length(smi):
    try:
        mol = _mol(smi)
        if mol is None:
            return np.nan
        patt = Chem.MolFromSmarts("[NX3,NX4;!$([N]~[!#6])]-[CX3](=[OX1])")
        n = len(mol.GetSubstructMatches(patt))
        return float(n + 1) if n > 0 else 0.0
    except Exception:
        return np.nan

def _is_alkyl_c(atom):
    return (
        atom.GetAtomicNum() == 6
        and not atom.GetIsAromatic()
        and not atom.IsInRing()
        and atom.GetHybridization() == Chem.rdchem.HybridizationType.SP3
    )

def get_long_alkyl_chain(smi, min_len=8):
    try:
        mol = _mol(smi)
        if mol is None:
            return np.nan

        cidxs = [a.GetIdx() for a in mol.GetAtoms() if _is_alkyl_c(a)]
        cset = set(cidxs)

        if not cidxs:
            return 0.0

        max_l = 0
        for start in cidxs:
            stack = [(start, frozenset([start]))]
            while stack:
                cur, vis = stack.pop()
                max_l = max(max_l, len(vis))

                for nb in mol.GetAtomWithIdx(cur).GetNeighbors():
                    ni = nb.GetIdx()
                    if ni in cset and ni not in vis:
                        stack.append((ni, vis | frozenset([ni])))

        return float(max_l if max_l >= min_len else 0)

    except Exception:
        return np.nan

sig = lambda p: "**" if p < 0.01 else ("*" if p < 0.05 else "  ")


# ══════════════════════════════════════════════════════════
#  Latent 추출
#  중요: 현재 encoder는 FP branch + SELFIES CNN branch라서
#        model.encode(..., tokens=...)로 넣어야 함.
# ══════════════════════════════════════════════════════════

bg_cond_vecs = torch.stack([ds_bg[i]["cond"] for i in range(len(ds_bg))]).to(DEVICE)
bg_fp_vecs   = torch.stack([ds_bg[i]["fp"] for i in range(len(ds_bg))]).to(DEVICE)
bg_tokens    = torch.stack([ds_bg[i]["tokens"] for i in range(len(ds_bg))]).to(DEVICE)

Z_bg_list = []

with torch.no_grad():
    for i in range(0, len(bg_fp_vecs), 256):
        mu, _ = model.encode(
            bg_fp_vecs[i:i+256],
            bg_cond_vecs[i:i+256],
            tokens=bg_tokens[i:i+256],
        )
        Z_bg_list.append(mu.cpu().numpy())

Z_bg_lat = np.concatenate(Z_bg_list, axis=0)

with torch.no_grad():
    Z_ms_lat, _ = model.encode(ms_fps, ms_cond, tokens=ms_tokens)
    Z_ms_lat = Z_ms_lat.cpu().numpy()

Z_all_lat = np.concatenate([Z_bg_lat, Z_ms_lat], axis=0)
n_bg = len(Z_bg_lat)

def gz(n):
    return Z_ms_lat[ms_idx[n]]

print(f"Latent extraction done: bg={n_bg}, ms={len(Z_ms_lat)}")


# ══════════════════════════════════════════════════════════
#  Vector Analysis
# ══════════════════════════════════════════════════════════

v_evo  = gz("Semaglutide")  - gz("GLP-1(7-36)")
v_dual = gz("Tirzepatide")  - gz("Semaglutide")
v_fail = gz("Taspoglutide") - gz("Semaglutide")

ce_d = cosine(v_evo, v_dual)
cf_d = cosine(v_evo, v_fail)
cd_d = cosine(v_dual, v_fail)

print("=" * 55)
print("  Vector Analysis")
print("=" * 55)
print(f"  Evo<->Dual : {ce_d:+.4f}  {'OK' if ce_d > 0 else 'NG'}")

print("\n[Pairwise distances]")
for a, b, note in [
    ("Semaglutide", "Taspoglutide", "Aib35 diff"),
    ("Semaglutide", "Tirzepatide",  "Evo->Dual"),
    ("Semaglutide", "Retatrutide",  "triple agonist"),
    ("Tirzepatide", "Retatrutide",  "poly-agonist cluster"),
    ("Albiglutide", "Dulaglutide",  "protein fusion"),
    ("GLP-1(7-36)", "GIP(1-42)",    "receptor selectivity"),
    ("Liraglutide", "Semaglutide",  "fatty acid C16->C18"),
]:
    if a in ms_idx and b in ms_idx:
        d = np.linalg.norm(gz(a) - gz(b))
        print(f"  {a:<15} <-> {b:<15} {d:7.3f}  {note}")

Latent extraction done: bg=1862, ms=12
  Vector Analysis
  Evo<->Dual : +0.2239  OK

[Pairwise distances]
  Semaglutide     <-> Taspoglutide      1.831  Aib35 diff
  Semaglutide     <-> Tirzepatide       3.055  Evo->Dual
  Semaglutide     <-> Retatrutide       2.658  triple agonist
  Tirzepatide     <-> Retatrutide       0.803  poly-agonist cluster
  Albiglutide     <-> Dulaglutide       0.800  protein fusion
  GLP-1(7-36)     <-> GIP(1-42)         4.428  receptor selectivity
  Liraglutide     <-> Semaglutide       1.282  fatty acid C16->C18


In [ ]:
# ══════════════════════════════════════════════════════════
#  Reconstruction Test
# ══════════════════════════════════════════════════════════
model.load_state_dict(state_dict)
model.eval()
print(f"Loaded ep{cp}")

from rdkit.Chem import DataStructs

print("=" * 100)
print(f"  {'Name':<18} {'Valid':>6} {'Tani':>8}  SMILES (60char)")
print("=" * 100)

with torch.no_grad():
    mu, _ = model.encode(ms_fps, ms_cond, tokens=ms_tokens)
    for b in range(N_MS):
        ids = model.smi_decoder.greedy(mu[b:b+1], ms_cond[b:b+1])
        recon_smi, valid = ids_to_smiles(ids)

        if valid:
            orig_smi = df_ms.loc[b, "smiles"]
            mol_o = Chem.MolFromSmiles(orig_smi)
            mol_r = Chem.MolFromSmiles(recon_smi)
            if mol_o and mol_r:
                fp_o = AllChem.GetMorganFingerprintAsBitVect(mol_o, RADIUS, N_BITS_MG)
                fp_r = AllChem.GetMorganFingerprintAsBitVect(mol_r, RADIUS, N_BITS_MG)
                tani = DataStructs.TanimotoSimilarity(fp_o, fp_r)
            else:
                tani = float("nan")
        else:
            tani = float("nan")

        mark = "✅" if valid else "❌"
        tani_str = f"{tani:.3f}" if not np.isnan(tani) else "  N/A"
        print(f"  {ms_names[b]:<18} {mark:>6} {tani_str:>8}  {(recon_smi or 'INVALID')[:800]}")

Loaded ep500
  Name                Valid     Tani  SMILES (60char)
  GLP-1(7-36)             ✅    0.662  CCC(C)C(NC(=O)C(CC1=CC=CC=C1)NC(=O)C(CCC(=O)O)NC(=O)C(CCCCN)NC(=O)C(C)NC(=O)C(C)NC(=O)C(CCC(N)=O)NC(=O)CNC(=O)C(CCC(=O)O)NC(=O)C(CC(C)C)NC(=O)C(CC2=CC=C(O)C=C2)NC(=O)C(CO)NC(=O)C(CO)NC(=O)C(NC(=O)C(CC(=O)O)NC(=O)C(CO)NC(=O)C(NC(=O)C(CC3=CC=CC=C3)NC(=O)C(NC(=O)CNC(=O)C(CCC(=O)O)NC(=O)CNC(=O)C(N)CC4=C[NH1]C=N4)C(C)O)C(C)O)C(C)C)C(=O)NC(C)C(=O)NC(CC5=C[NH1]C6=CC=CC=C56)C(=O)NC(CC(C)C)C(=O)NC(C(=O)NC(CCCCN)C(=O)NCC(=O)NC(CCCNC(=N)N)C(=O)O)C(C)C
  GIP(1-42)               ✅    1.000  CCC(C)C(NC(=O)C(CO)NC(=O)C(CC1=CC=C(O)C=C1)NC(=O)C(CC(=O)O)NC(=O)C(CO)NC(=O)C(NC(=O)C(CC2=CC=CC=C2)NC(=O)C(NC(=O)CNC(=O)C(CCC(=O)O)NC(=O)C(C)NC(=O)C(N)CC3=CC=C(O)C=C3)C(C)O)C(C)CC)C(=O)NC(C)C(=O)NC(CCSC)C(=O)NC(CC(=O)O)C(=O)NC(CCCCN)C(=O)NC(C(=O)NC(CC4=C[NH1]C=N4)C(=O)NC(CCC(N)=O)C(=O)NC(CCC(N)=O)C(=O)NC(CC(=O)O)C(=O)NC(CC5=CC=CC=C5)C(=O)NC(C(=O)NC(CC(N)=O)C(=O)NC(CC6=C[NH1]C7=CC=CC=C67)C(=O)NC(CC(C)C)C(=O)NC

In [ ]:
# ══════════════════════════════════════════════════════════
#  UMAP + PCA
# ══════════════════════════════════════════════════════════
# ── 상수 ───────────────────────────────────────────────────
def _cond_to_group_idx(cond):
    if cond == [0, 1, 0]:
        return 1   # GCGR
    if cond == [0, 0, 1]:
        return 2   # GIP
    return 0       # GLP1

MS_RECEPTOR = {
    name: _cond_to_group_idx(cond)
    for name, cond in MS_COND_VEC.items()
}

COND_PALETTE = ["#27ae60", "#e67e22", "#2980b9"]
COND_NAMES   = COND_LABELS
EVO_EXCLUDE  = {"GIP(1-42)", "Glucagon(1-29)"}
GEN_COLOR    = {"native": "#7f8c8d", "Gen1": "#3498db", "Gen2": "#9b59b6",
                "Gen3": "#27ae60",   "Gen4": "#e67e22",  "failed": "#e74c3c"}
GEN_MARKER   = {"native": "D", "Gen1": "o", "Gen2": "s",
                "Gen3": "o",  "Gen4": "^", "failed": "X"}
MS_GEN = {
    "GLP-1(7-36)":    "native", "Exenatide":   "Gen1", "Lixisenatide": "Gen1",
    "Liraglutide":    "Gen2",   "Albiglutide": "Gen2", "Dulaglutide":  "Gen2",
    "Semaglutide":    "Gen3",   "Tirzepatide": "Gen4", "Retatrutide":  "Gen4",
    "Taspoglutide":   "failed", "Glucagon(1-29)": "native", "GIP(1-42)": "native",
}
GEN_ORDER = ["native", "Gen1", "Gen2", "Gen3", "Gen4"]

# ── 보조 함수 ──────────────────────────────────────────────
def ms_cond_color(name):
    return COND_PALETTE[MS_RECEPTOR.get(name, 0)]

def draw_ms_umap(ax, size=220, exclude=None):
    for name in ms_names:
        if name not in ms_idx: continue
        if exclude and name in exclude: continue
        i = ms_idx[name]
        ax.scatter(Z_ms_2d[i, 0], Z_ms_2d[i, 1],
                   c=ms_cond_color(name),
                   marker="*", s=size, edgecolors="white", lw=1.2, zorder=6)

bg_cond_cpu = bg_cond_vecs.argmax(dim=-1).cpu().numpy()

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.3, spread=1.0,
                    metric="cosine", random_state=SEED, n_jobs=1)
Z_2d    = reducer.fit_transform(Z_all_lat)
Z_bg_2d = Z_2d[:n_bg]
Z_ms_2d = Z_2d[n_bg:]

print("Running ms-only PCA...")
pca_ms    = PCA(n_components=2, random_state=SEED)
Z_ms_only = pca_ms.fit_transform(Z_ms_lat)
var_ratio = pca_ms.explained_variance_ratio_
print(f"PC1={var_ratio[0]:.1%}  PC2={var_ratio[1]:.1%}")

# ── 세대별 centroid 계산 ───────────────────────────────────
gen_centroids = {}
for gen in GEN_ORDER + ["failed"]:
    members = [n for n, g in MS_GEN.items()
               if g == gen and n in ms_idx and n not in EVO_EXCLUDE]
    if not members: continue
    pts = np.array([Z_ms_only[ms_idx[n]] for n in members])
    gen_centroids[gen] = pts.mean(axis=0)

from matplotlib.lines import Line2D

PLOT_GEN = MS_GEN.copy()
PLOT_GEN["Exenatide"]    = "Exendin-4 analogs"
PLOT_GEN["Lixisenatide"] = "Exendin-4 analogs"
PLOT_GEN["Liraglutide"]  = "GLP-1R mono agonists"
PLOT_GEN["Albiglutide"]  = "GLP-1R mono agonists"
PLOT_GEN["Dulaglutide"]  = "GLP-1R mono agonists"
PLOT_GEN["Semaglutide"]  = "GLP-1R mono agonists"
PLOT_GEN["Taspoglutide"] = "Failed candidate"
PLOT_GEN["Tirzepatide"]  = "Dual agonist"
PLOT_GEN["Retatrutide"]  = "Triple agonist"

PLOT_COLOR = {
    "Exendin-4 analogs":    "#3498db",
    "GLP-1R mono agonists": "#27ae60",
    "Dual agonist":         "#e67e22",
    "Triple agonist":       "#e67e22",
    "Failed candidate":     "#e74c3c",
    "native":               "#888780",
}

PLOT_MARKER = {
    "Exendin-4 analogs":    "o",
    "GLP-1R mono agonists": "o",
    "Dual agonist":         "^",
    "Triple agonist":       "D",
    "Failed candidate":     "X",
    "native":               "D",
}

# centroid 색도 통일
GEN_COLOR_PLOT = GEN_COLOR.copy()
GEN_COLOR_PLOT.update({
    "Gen1":   "#3498db",
    "Gen2":   "#27ae60",
    "Gen3":   "#27ae60",
    "Gen4":   "#e67e22",
    "failed": "#e74c3c",
    "native": "#888780",
})

fig, ax = plt.subplots(figsize=(10, 8))

# ── 개별 milestone 점 ────────────────────────────────────
for name in ms_names:
    if name not in ms_idx or name in EVO_EXCLUDE:
        continue
    i = ms_idx[name]
    plot_group = PLOT_GEN.get(name, "GLP-1R mono agonists")
    ax.scatter(
        Z_ms_only[i, 0], Z_ms_only[i, 1],
        c=PLOT_COLOR[plot_group],
        marker=PLOT_MARKER[plot_group],
        s=200, edgecolors="white", lw=1.5, zorder=5, alpha=0.7
    )

# ── 이름표 ────────────────────────────────────────────────
NAME_OFFSET = {
    "GLP-1(7-36)":  (+0.05, +0.08),
    "Exenatide":    (+0.05, +0.08),
    "Lixisenatide": (+0.05, -0.12),
    "Liraglutide":  (+0.05, +0.08),
    "Albiglutide":  (+0.05, +0.08),
    "Dulaglutide":  (+0.05, +0.08),
    "Semaglutide":  (+0.05, +0.08),
    "Taspoglutide": (-0.05, +0.08),
    "Tirzepatide":  (+0.05, -0.12),
    "Retatrutide":  (+0.05, +0.08),
}

for name in ms_names:
    if name not in ms_idx or name in EVO_EXCLUDE:
        continue
    i = ms_idx[name]
    plot_group = PLOT_GEN.get(name, "GLP-1R mono agonists")
    dx, dy = NAME_OFFSET.get(name, (+0.05, +0.08))
    x, y = Z_ms_only[i, 0] + dx, Z_ms_only[i, 1] + dy
    ax.text(
        x, y, name,
        fontsize=9, fontweight="bold",
        color=PLOT_COLOR[plot_group],
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
        zorder=9
    )

# ── centroid 점 ───────────────────────────────────────────
for gen, pt in gen_centroids.items():
    ax.scatter(
        pt[0], pt[1],
        c=GEN_COLOR_PLOT.get(gen, "#888780"),
        marker=GEN_MARKER[gen],
        s=450, edgecolors="black", lw=2, zorder=8
    )

# ── 진화 방향 화살표 ──────────────────────────────────────
gen_path = [g for g in GEN_ORDER if g in gen_centroids and g != "native"]
for i in range(len(gen_path) - 1):
    a, b = gen_path[i], gen_path[i + 1]
    ax.annotate(
        "",
        xy=tuple(gen_centroids[b][:2]),
        xytext=tuple(gen_centroids[a][:2]),
        arrowprops=dict(
            arrowstyle="-|>", color="#2c3e50",
            lw=2.5, mutation_scale=20,
            connectionstyle="arc3,rad=0.1"
        ),
        zorder=6
    )

ax.set_title("GLP-1 Drug Trajectory  PCA", fontsize=15, fontweight="bold")
# ax.text(0.5, 1.01,
#         f"PC1 = {var_ratio[0]:.1%}   PC2 = {var_ratio[1]:.1%}",
#         transform=ax.transAxes, ha="center", fontsize=9, color="gray")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")

ax.set_xlim(Z_ms_only[:, 0].min() - 0.3, Z_ms_only[:, 0].max() + 0.5)
ax.set_ylim(Z_ms_only[:, 1].min() - 0.3, Z_ms_only[:, 1].max() + 0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/pca_centroid_v4.png", dpi=200)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════
#  Chemical Property Analysis
# ══════════════════════════════════════════════════════════

ALKYL_MIN = 8
bg_smiles = [ds_bg.valid_rows[i]["smiles"] for i in range(len(ds_bg))]

props = {
    "peptide_len": ("Peptide length proxy", get_peptide_length),
    "alkyl_chain": ("Long alkyl chain proxy", lambda s: get_long_alkyl_chain(s, ALKYL_MIN)),
    "mol_weight":  ("MW (Da)",               get_mw),
    "logp":        ("LogP",                  get_logp),
    "tpsa":        ("TPSA",                  get_tpsa),
}

print("\nCalculating chemical properties...")
bg_props = {}
ms_props = {}
for key, (label, fn) in props.items():
    bg_props[key] = np.array([fn(s) for s in bg_smiles], dtype=float)
    ms_props[key] = {n: fn(df_ms.loc[ms_idx[n], "smiles"])
                     for n in ms_names if n in ms_idx}
print("Done.")

# Spearman correlation (latent vs property)
print("\n" + "=" * 70)
print("  Latent-Property Spearman Correlation (UMAP)")
print("=" * 70)
print(f"  {'Property':<25} {'UMAP1 rho':>9} {'p':>8} {'UMAP2 rho':>9} {'p':>8}")
print("-" * 70)
for key, (label, _) in props.items():
    vals = bg_props[key]; mask = ~np.isnan(vals)
    if mask.sum() < 10: continue
    r1, p1 = spearmanr(Z_bg_2d[mask, 0], vals[mask])
    r2, p2 = spearmanr(Z_bg_2d[mask, 1], vals[mask])
    print(f"  {label:<25} {r1:>+9.3f}{sig(p1)} {p1:>8.4f} {r2:>+9.3f}{sig(p2)} {p2:>8.4f}")

# Milestone chemical property table
print("\n" + "=" * 80)
print("  Milestone Chemical Properties")
print("=" * 80)
print(f"  {'Name':<18} {'AA':>5} {'AlkylC':>7} {'MW':>8} {'LogP':>7} {'TPSA':>7}")
print("-" * 80)
for n in ms_names:
    if n not in ms_idx: continue
    smi   = df_ms.loc[ms_idx[n], "smiles"]
    fmt   = lambda v, d=0: f"{v:.{d}f}" if not np.isnan(v) else "N/A"
    print(f"  {n:<18} {fmt(get_peptide_length(smi)):>5} "
          f"{fmt(get_long_alkyl_chain(smi, ALKYL_MIN)):>7} "
          f"{fmt(get_mw(smi), 0):>8} "
          f"{fmt(get_logp(smi), 1):>7} "
          f"{fmt(get_tpsa(smi), 0):>7}")

fig, ax = plt.subplots(figsize=(8, 8))

# ── Background: 전체 데이터 기준 ─────────────────────────
vals = bg_props["peptide_len"]
nv = ~np.isnan(vals)

# 전체 데이터 기준 color scale
vmin_v = np.nanpercentile(vals[nv], 5)
vmax_v = np.nanpercentile(vals[nv], 95)

sc = ax.scatter(
    Z_bg_2d[nv, 0], Z_bg_2d[nv, 1],
    c=vals[nv],
    cmap="viridis_r",
    vmin=vmin_v,
    vmax=vmax_v,
    s=6,
    alpha=0.6,
    edgecolors="none",
    zorder=2
)

cbar = plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.01)
cbar.set_label("Peptide Length (proxy)", fontsize=9)

# ── 전체 UMAP 범위 표시 ─────────────────────────────────
x_all = np.concatenate([Z_bg_2d[nv, 0], Z_ms_2d[:, 0]])
y_all = np.concatenate([Z_bg_2d[nv, 1], Z_ms_2d[:, 1]])

ax.set_xlim(x_all.min() - 0.8, x_all.max() + 0.8)
ax.set_ylim(y_all.min() - 0.8, y_all.max() + 0.8)

# ── 수동 오프셋 ───────────────────────────────────────────
OFFSETS = {
    # 상단 GLP-1 cluster
    "GLP-1(7-36)":    ( 0.25,  2.0),
    "Albiglutide":    ( 0.55,  1.0),
    "Dulaglutide":    ( 0.65, 0.0),

    # 좌측 lipidated / GLP-1 analog cluster
    "Liraglutide":    (-1.70,  2.0),
    "Taspoglutide":   (-5.0,  0.10),
    "Semaglutide":    (-1.85, -2.0),

    # 중앙 dual/triple cluster
    "Retatrutide":    (-2.10,  2.0),
    "Tirzepatide":    (-2.25, -2.0),

    # 중앙-right native/incretin cluster
    "GIP(1-42)":      ( 0.45,  0.65),
    "Glucagon(1-29)": ( 1.0,  0.35),
    "Lixisenatide":   ( 0.55,  0.05),
    "Exenatide":      ( 1.15, -1.0),
}
# ── Milestone 포인트 + 이름 표시 ─────────────────────────
for name in ms_names:
    if name not in ms_idx:
        continue

    i = ms_idx[name]
    x, y = Z_ms_2d[i, 0], Z_ms_2d[i, 1]
    dx, dy = OFFSETS.get(name, (0.3, 0.3))

    ax.scatter(
        x, y,
        color="#1e3f5a",
        marker="o",
        s=70,
        edgecolors="white",
        linewidths=1.2,
        zorder=7
    )

    ax.annotate(
        name,
        xy=(x, y),
        xytext=(x + dx, y + dy),
        fontsize=8,
        color="#1a1a2e",
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.2",
            fc="white",
            ec="#cccccc",
            alpha=0.9,
            lw=0.5
        ),
        arrowprops=dict(
            arrowstyle="-",
            color="#555",
            lw=0.8
        ),
        zorder=8
    )


ax.set_xlabel("UMAP-1", fontsize=11)
ax.set_ylabel("UMAP-2", fontsize=11)
ax.set_title(
    "Latent Space UMAP\n(colored by Peptide Length)",
    fontsize=11,
    fontweight="bold"
)

ax.grid(alpha=0.15, color="gray")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/umap_milestone_full_v7.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved.")

In [ ]:
# ══════════════════════════════════════════════════════════
#  cAMP / Bias Activity Analysis
# ══════════════════════════════════════════════════════════
from scipy.stats import spearmanr
import numpy as np
import matplotlib.pyplot as plt

print("\n" + "=" * 70)
print("  cAMP Activity vs Latent Spearman Correlation")
print("=" * 70)

# ── cAMP mask ───────────────────────────────────────────
camp_mask = ~np.isnan(y_camp)

# pEC50 = 5.0은 lower-bound / inactive placeholder 성격이 강하므로 제외
camp_mask_clean = camp_mask & (y_camp != 5.0)

print(f"  cAMP total             : {camp_mask.sum()}")
print(f"  cAMP used (excl 5.0)   : {camp_mask_clean.sum()}")
print()

# ── cAMP vs UMAP Spearman ───────────────────────────────
if camp_mask_clean.sum() > 10:
    r1c, p1c = spearmanr(Z_bg_2d[camp_mask_clean, 0], y_camp[camp_mask_clean])
    r2c, p2c = spearmanr(Z_bg_2d[camp_mask_clean, 1], y_camp[camp_mask_clean])

    print(f"  cAMP pEC50 (excl 5.0, n={camp_mask_clean.sum()})")
    print(f"    UMAP1 rho={r1c:+.3f}{sig(p1c)}")
    print(f"    UMAP2 rho={r2c:+.3f}{sig(p2c)}")
else:
    print("  cAMP data insufficient -> skip")


if camp_mask_clean.sum() > 10:
    fig, ax = plt.subplots(figsize=(6, 6))

    vals = y_camp
    nv = camp_mask_clean

    vmin_v = np.nanpercentile(vals[nv], 5)
    vmax_v = np.nanpercentile(vals[nv], 95)

    sc = ax.scatter(
        Z_bg_2d[nv, 0], Z_bg_2d[nv, 1],
        c=vals[nv],
        cmap="viridis",
        vmin=vmin_v,
        vmax=vmax_v,
        s=8,
        alpha=0.65,
        edgecolors="none",
        zorder=2
    )

    cbar = plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.01)
    cbar.set_label("cAMP pEC50", fontsize=9)

    # milestone points
    for name in ms_names:
        if name not in ms_idx:
            continue

        i = ms_idx[name]
        x, y = Z_ms_2d[i, 0], Z_ms_2d[i, 1]

        ax.scatter(
            x, y,
            color="#1e3f5a",
            marker="o",
            s=70,
            edgecolors="white",
            linewidths=1.2,
            zorder=7
        )

    ax.set_xlabel("UMAP-1", fontsize=11)
    ax.set_ylabel("UMAP-2", fontsize=11)
    ax.set_title(
        "Latent Space UMAP\n(colored by cAMP pEC50)",
        fontsize=11,
        fontweight="bold"
    )
    ax.grid(alpha=0.15, color="gray")

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/umap_camp_pec50.png", dpi=200, bbox_inches="tight")
    plt.show()

    print(f"  Saved: {OUT_DIR}/umap_camp_pec50.png")


In [ ]:
# ══════════════════════════════════════════════════════════
#  Distance Matrix + Tanimoto Matrix
# ══════════════════════════════════════════════════════════
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── Milestone latent vectors ──────────────────────────────
names_plot = [n for n in ms_names if n in ms_idx]
Z_ms_plot  = np.array([Z_ms_lat[ms_idx[n]] for n in names_plot])
dist_mat   = cdist(Z_ms_plot, Z_ms_plot, metric="euclidean")

# ── Short labels ──────────────────────────────────────────
short = [
    n.replace("(7-36)", "").replace("(1-42)", "").replace("(1-29)", "").strip()
    for n in names_plot
]

mask_diag = np.eye(len(names_plot), dtype=bool)

# ── 그룹별 색상 (y축 레이블) ──────────────────────────────
GROUP_COLOR = {
    "GLP-1":        "#4e9af1",
    "GIP":          "#f4a261",
    "Glucagon":     "#f4a261",
    "Exenatide":    "#4e9af1",
    "Lixisenatide": "#4e9af1",
    "Liraglutide":  "#4e9af1",
    "Albiglutide":  "#4e9af1",
    "Taspoglutide": "#4e9af1",
    "Dulaglutide":  "#4e9af1",
    "Semaglutide":  "#4e9af1",
    "Tirzepatide":  "#a8d8a8",
    "Retatrutide":  "#9b59b6",
}

fig, ax = plt.subplots(figsize=(9, 8))

sns.heatmap(
    dist_mat,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd_r",
    xticklabels=short,
    yticklabels=short,
    mask=mask_diag,
    ax=ax,
    linewidths=0.4,
    linecolor="white",
    square=True,
    annot_kws={"fontsize": 8},
    cbar_kws={"label": "Euclidean Distance", "shrink": 0.75}
)

ax.set_title("Latent Space Distance Matrix", fontsize=11, fontweight="bold", pad=10)
ax.tick_params(axis="x", labelrotation=45, labelsize=7)
ax.tick_params(axis="y", labelrotation=0, labelsize=7)

# y축 레이블 색상
for tick, name in zip(ax.get_yticklabels(), short):
    tick.set_color(GROUP_COLOR.get(name, "black"))
    tick.set_fontweight("bold")
for tick, name in zip(ax.get_xticklabels(), short):
    tick.set_color(GROUP_COLOR.get(name, "black"))
    tick.set_fontweight("bold")

# 범례
import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(color="#4e9af1", label="GLP-1R agonist"),
    mpatches.Patch(color="#f4a261", label="GCGR / GIP native"),
    mpatches.Patch(color="#a8d8a8", label="Dual agonist"),
    mpatches.Patch(color="#9b59b6", label="Triple agonist"),
]
ax.legend(handles=legend_patches, loc="upper left",
          bbox_to_anchor=(1.18, 1.0), fontsize=8,
          framealpha=1.0, title="Receptor Class", title_fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/distance_matrix_only.png", dpi=250, bbox_inches="tight")
plt.show()
print("Saved.")

In [ ]:
# ══════════════════════════════════════════════════════════
#  Latent Interpolation Analysis
#  - FP decoder trajectory
#  - Deterministic SELFIES decoding trajectory
#  - Stochastic sampling trajectory
# ══════════════════════════════════════════════════════════

from rdkit.Chem import rdMolDescriptors
from scipy.stats import spearmanr
BEAM_WIDTH = 5

INTERP_SOURCE = "Semaglutide"
INTERP_TARGET = "Taspoglutide"
INTERP_STEPS  = 11
INTERP_COND_MODE = "interp"   # "source", "target", "interp"

INTERP_SAMPLE_KWARGS = dict(
    temperature=0.8,
    top_k=10,
    top_p=0.90,
    repetition_penalty=1.3,
    eos_boost=0.005,
)

INTERP_MIN_LEN = 30
INTERP_MAX_LEN = 120
INTERP_SAMPLE_N = 30


def peptide_bond_count(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return 0
    patt = Chem.MolFromSmarts("C(=O)N")
    return len(mol.GetSubstructMatches(patt))


def mol_props(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    return {
        "mw": Descriptors.MolWt(mol),
        "logp": Descriptors.MolLogP(mol),
        "tpsa": rdMolDescriptors.CalcTPSA(mol),
        "amide": peptide_bond_count(smi),
        "rot_bonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "heavy_atoms": mol.GetNumHeavyAtoms(),
    }


def alkyl_proxy(smi):
    return get_long_alkyl_chain(smi, min_len=8)


def fp_tanimoto_vec(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / (union + 1e-8)


def get_interp_cond(alpha, src_cond, tgt_cond, cond_mode):
    if cond_mode == "source":
        return src_cond
    if cond_mode == "target":
        return tgt_cond
    if cond_mode == "interp":
        return (1 - alpha) * src_cond + alpha * tgt_cond
    raise ValueError("cond_mode must be source, target, or interp")

def milestone_props(names):
    rows = []
    for name in names:
        i = ms_idx[name]
        smi = df_ms.iloc[i]["smiles"]
        p = mol_props(smi)
        if p is None:
            continue
        p["name"] = name
        p["alkyl_proxy"] = alkyl_proxy(smi)
        p["tokens"] = int((ds_ms[i]["tokens"] != PAD_IDX).sum().item())
        rows.append(p)

    return pd.DataFrame(rows)[[
        "name", "mw", "logp", "tpsa", "amide",
        "alkyl_proxy", "rot_bonds", "heavy_atoms", "tokens"
    ]]


def interpolate_fp_decoder(source_name, target_name, n_steps=11,
                           cond_mode="interp", threshold=0.5):
    ctx = get_milestone_latents(source_name, target_name)

    src_bits = ctx["src_fp"].cpu().numpy()[0] > 0.5
    tgt_bits = ctx["tgt_fp"].cpu().numpy()[0] > 0.5

    rows = []
    for alpha in np.linspace(0, 1, n_steps):
        z = (1 - alpha) * ctx["z_src"] + alpha * ctx["z_tgt"]
        cond = get_interp_cond(alpha, ctx["src_cond"], ctx["tgt_cond"], cond_mode)

        with torch.no_grad():
            fp_hat = model.fp_decoder(z, cond).cpu().numpy()[0]

        bits = fp_hat > threshold
        rows.append({
            "alpha": alpha,
            "sim_fp_source": fp_tanimoto_vec(bits, src_bits),
            "sim_fp_target": fp_tanimoto_vec(bits, tgt_bits),
            "fp_on_bits": int(bits.sum()),
            "fp_prob_mean": float(fp_hat.mean()),
        })

    return pd.DataFrame(rows)


def decode_interp_deterministic(source_name, target_name, n_steps=11,
                                method="beam", cond_mode="interp"):
    ctx = get_milestone_latents(source_name, target_name)

    rows = []
    for alpha in np.linspace(0, 1, n_steps):
        z = (1 - alpha) * ctx["z_src"] + alpha * ctx["z_tgt"]
        cond = get_interp_cond(alpha, ctx["src_cond"], ctx["tgt_cond"], cond_mode)

        if method == "beam":
            ids = model.smi_decoder.beam_search(
                z, cond, beam_width=BEAM_WIDTH, max_len=INTERP_MAX_LEN
            )
        elif method == "greedy":
            ids = model.smi_decoder.greedy(z, cond, max_len=INTERP_MAX_LEN)
        else:
            raise ValueError("method must be greedy or beam")

        smi = decode_tokens(ids)
        p = mol_props(smi) if check_valid(smi) else None

        if p is None:
            rows.append({
                "alpha": alpha,
                "valid": False,
                "method": method,
                "smiles": smi,
            })
            continue

        rows.append({
            "alpha": alpha,
            "valid": True,
            "method": method,
            **p,
            "alkyl_proxy": get_long_alkyl_chain(smi, min_len=8),
            "sim_source": fp_tanimoto(smi, ctx["src_smi"]),
            "sim_target": fp_tanimoto(smi, ctx["tgt_smi"]),
            "n_tokens": len(ids),
            "smiles": smi,
        })

    return pd.DataFrame(rows)



def summarize_interp_samples(df_interp):
    if len(df_interp) == 0:
        return pd.DataFrame()

    return df_interp.groupby("alpha").agg({
        "mw": "median",
        "logp": "median",
        "tpsa": "median",
        "amide": "median",
        "alkyl_proxy": "median",
        "rot_bonds": "median",
        "sim_source": "median",
        "sim_target": "median",
        "n_tokens": "median",
    }).reset_index()


def corr_with_alpha(df, cols, label=""):
    print(f"\n[Spearman vs alpha] {label}")
    rows = []
    for c in cols:
        if c not in df.columns:
            continue
        x = df["alpha"].values
        y = df[c].values
        ok = np.isfinite(y)
        if ok.sum() < 3:
            rho, p = np.nan, np.nan
        else:
            rho, p = spearmanr(x[ok], y[ok])
        rows.append({"metric": c, "rho": rho, "p": p})
        print(f"  {c:<15} rho={rho:+.3f} p={p:.4g}")
    return pd.DataFrame(rows)


# ── 보조 함수 ──────────────────────────────────────────────
def decode_tokens(ids):
    smi, _ = ids_to_smiles(ids)
    return smi

def check_valid(smi):
    if not smi: return False
    return Chem.MolFromSmiles(smi) is not None

def fp_tanimoto(smi_a, smi_b):
    mol_a = Chem.MolFromSmiles(str(smi_a))
    mol_b = Chem.MolFromSmiles(str(smi_b))
    if mol_a is None or mol_b is None: return np.nan
    fp_a = AllChem.GetMorganFingerprintAsBitVect(mol_a, RADIUS, N_BITS_MG)
    fp_b = AllChem.GetMorganFingerprintAsBitVect(mol_b, RADIUS, N_BITS_MG)
    return DataStructs.TanimotoSimilarity(fp_a, fp_b)

def get_milestone_latents(source_name, target_name):
    assert source_name in ms_idx, f"Unknown source: {source_name}"
    assert target_name in ms_idx, f"Unknown target: {target_name}"

    src_i = ms_idx[source_name]
    tgt_i = ms_idx[target_name]

    src_fp   = ms_fps[src_i:src_i+1]
    tgt_fp   = ms_fps[tgt_i:tgt_i+1]
    src_cond = ms_cond[src_i:src_i+1]
    tgt_cond = ms_cond[tgt_i:tgt_i+1]

    model.eval()
    with torch.no_grad():
        src_tok = ms_tokens[src_i:src_i+1]
        tgt_tok = ms_tokens[tgt_i:tgt_i+1]
        z_src, _ = model.encode(src_fp, src_cond, tokens=src_tok)
        z_tgt, _ = model.encode(tgt_fp, tgt_cond, tokens=tgt_tok)

    return {
        "src_i": src_i, "tgt_i": tgt_i,
        "src_fp": src_fp, "tgt_fp": tgt_fp,
        "src_cond": src_cond, "tgt_cond": tgt_cond,
        "z_src": z_src, "z_tgt": z_tgt,
        "src_smi": df_ms.iloc[src_i]["smiles"],  # loc → iloc
        "tgt_smi": df_ms.iloc[tgt_i]["smiles"],
    }

def decode_interp_sample(source_name, target_name, n_steps=11, n_per_step=20,
                         cond_mode="interp", sample_kwargs=None):
    if sample_kwargs is None:
        sample_kwargs = INTERP_SAMPLE_KWARGS

    ctx = get_milestone_latents(source_name, target_name)

    rows = []
    for alpha in np.linspace(0, 1, n_steps):
        z = (1 - alpha) * ctx["z_src"] + alpha * ctx["z_tgt"]
        cond = get_interp_cond(alpha, ctx["src_cond"], ctx["tgt_cond"], cond_mode)

        with torch.no_grad():
            for j in range(n_per_step):
                ids = model.smi_decoder.sample(
                    z, cond,
                    max_len=INTERP_MAX_LEN,
                    min_len=INTERP_MIN_LEN,
                    **sample_kwargs,
                )
                smi = decode_tokens(ids)
                p = mol_props(smi) if check_valid(smi) else None
                if p is None:
                    continue

                rows.append({
                    "alpha": alpha,
                    "sample_id": j + 1,
                    "valid": True,
                    **p,
                    "alkyl_proxy": get_long_alkyl_chain(smi, min_len=8),
                    "sim_source": fp_tanimoto(smi, ctx["src_smi"]),
                    "sim_target": fp_tanimoto(smi, ctx["tgt_smi"]),
                    "n_tokens": len(ids),
                    "smiles": smi,
                })

    return pd.DataFrame(rows)

def print_pair_info(source_name, target_name):
    df_pair = milestone_props([source_name, target_name])

    print("\n[Milestone Properties]")
    print(df_pair.to_string(index=False))

    if len(df_pair) == 2:
        a = df_pair.iloc[0]
        b = df_pair.iloc[1]

        print("\n[Difference (target - source)]")
        print(
            f"MW={b['mw']-a['mw']:+.1f} | "
            f"logP={b['logp']-a['logp']:+.2f} | "
            f"TPSA={b['tpsa']-a['tpsa']:+.1f} | "
            f"Amide={b['amide']-a['amide']:+.0f} | "
            f"Alkyl={b['alkyl_proxy']-a['alkyl_proxy']:+.0f} | "
            f"RotBonds={b['rot_bonds']-a['rot_bonds']:+.0f} | "
            f"HeavyAtoms={b['heavy_atoms']-a['heavy_atoms']:+.0f} | "
            f"Tokens={b['tokens']-a['tokens']:+.0f}"
        )

def run_interp(source_name, target_name, cond_mode="source",
               n_steps=None, n_per_step=None, sample_kwargs=None):
    n_steps       = n_steps       or INTERP_STEPS
    n_per_step    = n_per_step    or INTERP_SAMPLE_N
    sample_kwargs = sample_kwargs or INTERP_SAMPLE_KWARGS

    print(f"\n{'='*80}")
    print(f"  {source_name} -> {target_name}  (cond_mode={cond_mode})")
    print(f"{'='*80}")


    print_pair_info(source_name, target_name)

    print("\n[FP Decoder]")
    df_fp = interpolate_fp_decoder(
        source_name, target_name,
        n_steps=n_steps,
        cond_mode=cond_mode,
    )
    print(df_fp.to_string(index=False))

    print("\n[Deterministic (beam)]")
    df_det = decode_interp_deterministic(
        source_name, target_name,
        n_steps=n_steps,
        method="beam",
        cond_mode=cond_mode,
    )
    print(df_det.drop(columns=["smiles"], errors="ignore").to_string(index=False))

    print("\n[Stochastic (sample)]")
    df_samp = decode_interp_sample(
        source_name, target_name,
        n_steps=n_steps,
        n_per_step=n_per_step,
        cond_mode=cond_mode,
        sample_kwargs=sample_kwargs,
    )
    df_summary = summarize_interp_samples(df_samp)
    print(df_summary.to_string(index=False))

    props = ["mw", "tpsa", "amide", "alkyl_proxy",
             "sim_source", "sim_target", "n_tokens"]
    corr_with_alpha(df_fp,
        ["sim_fp_source", "sim_fp_target", "fp_on_bits"],
        label="FP decoder")
    corr_with_alpha(df_summary, props, label="sample median")

    return df_fp, df_det, df_summary


# Exenatide → [Lira/Taspo/Albi/Dula centroid] → Semaglutide → Tirzepatide → Retatrutide
CLUSTER_NAMES = ["Liraglutide", "Taspoglutide", "Albiglutide", "Dulaglutide"]

# ── centroid z / cond 계산 ────────────────────────────────
model.eval()
with torch.no_grad():
    z_cluster_list = []
    for name in CLUSTER_NAMES:
        i = ms_idx[name]
        z, _ = model.encode(
            ms_fps[i:i+1], ms_cond[i:i+1],
            tokens=ms_tokens[i:i+1]
        )
        z_cluster_list.append(z)
    z_cluster = torch.stack(z_cluster_list).mean(dim=0)

cond_cluster = torch.stack([ms_cond[ms_idx[n]] for n in CLUSTER_NAMES]).mean(dim=0, keepdim=True)
centroid_tuple = (z_cluster, cond_cluster, None)

print(f"Cluster centroid: {CLUSTER_NAMES}")


# ── 물성 출력 함수 ────────────────────────────────────────
def get_props_dict(x):
    if isinstance(x, str):
        i = ms_idx[x]
        smi = df_ms.iloc[i]["smiles"]
        p = mol_props(smi)
        p["alkyl_proxy"] = get_long_alkyl_chain(smi, min_len=8)
        p["name"] = x
        return [p]
    else:
        rows = []
        for n in CLUSTER_NAMES:
            i = ms_idx[n]
            smi = df_ms.iloc[i]["smiles"]
            p = mol_props(smi)
            p["alkyl_proxy"] = get_long_alkyl_chain(smi, min_len=8)
            p["name"] = n
            rows.append(p)
        keys = ["mw","logp","tpsa","amide","alkyl_proxy","rot_bonds","heavy_atoms"]
        avg = {k: np.mean([r[k] for r in rows]) for k in keys}
        avg["name"] = f"[centroid avg]"
        return [avg]

def print_segment_info(src, tgt, label):
    print(f"\n[{label} — Milestone Properties]")
    rows = get_props_dict(src) + get_props_dict(tgt)
    df_info = pd.DataFrame(rows)[["name","mw","logp","tpsa","amide","alkyl_proxy","rot_bonds","heavy_atoms"]]
    print(df_info.to_string(index=False))

# ── latent 가져오기 ───────────────────────────────────────
def get_latent_tuple(x):
    if isinstance(x, str):
        i = ms_idx[x]
        with torch.no_grad():
            z, _ = model.encode(
                ms_fps[i:i+1], ms_cond[i:i+1],
                tokens=ms_tokens[i:i+1]
            )
        return z, ms_cond[i:i+1], df_ms.iloc[i]["smiles"]
    else:
        return x  # (z, cond, smi) tuple


# ── FP decoder 보간 ───────────────────────────────────────
def interp_fp_decoder_custom(src, tgt, n_steps=11, cond_mode="source"):
    z_src, cond_src, _ = get_latent_tuple(src)
    z_tgt, cond_tgt, _ = get_latent_tuple(tgt)

    # src/tgt FP (centroid면 평균)
    def get_fp_bits(x):
        if isinstance(x, str):
            i = ms_idx[x]
            with torch.no_grad():
                fp = model.fp_decoder(
                    *model.encode(ms_fps[i:i+1], ms_cond[i:i+1], tokens=ms_tokens[i:i+1])[:1],
                    ms_cond[i:i+1]
                ).cpu().numpy()[0]
            return fp > 0.5
        else:
            bits_list = []
            for n in CLUSTER_NAMES:
                i = ms_idx[n]
                with torch.no_grad():
                    z, _ = model.encode(ms_fps[i:i+1], ms_cond[i:i+1], tokens=ms_tokens[i:i+1])
                    fp = model.fp_decoder(z, ms_cond[i:i+1]).cpu().numpy()[0]
                bits_list.append(fp > 0.5)
            return np.mean(bits_list, axis=0) > 0.5

    src_bits = get_fp_bits(src)
    tgt_bits = get_fp_bits(tgt)

    rows = []
    for alpha in np.linspace(0, 1, n_steps):
        z    = (1 - alpha) * z_src + alpha * z_tgt
        cond = get_interp_cond(alpha, cond_src, cond_tgt, cond_mode)
        with torch.no_grad():
            fp_hat = model.fp_decoder(z, cond).cpu().numpy()[0]
        bits = fp_hat > 0.5
        rows.append({
            "alpha":         alpha,
            "sim_fp_source": fp_tanimoto_vec(bits, src_bits),
            "sim_fp_target": fp_tanimoto_vec(bits, tgt_bits),
            "fp_on_bits":    int(bits.sum()),
            "fp_prob_mean":  float(fp_hat.mean()),
        })
    return pd.DataFrame(rows)


# ── stochastic sampling 보간 ──────────────────────────────
def run_interp_custom(src, tgt, label, cond_mode="source", n_steps=11, n_per_step=30):
    z_src, cond_src, smi_src = get_latent_tuple(src)
    z_tgt, cond_tgt, smi_tgt = get_latent_tuple(tgt)

    print(f"\n{'='*70}")
    print(f"  {label}  (cond_mode={cond_mode})")
    print(f"{'='*70}")

    print_segment_info(src, tgt, label)

    # FP decoder
    print("\n[FP Decoder]")
    df_fp = interp_fp_decoder_custom(src, tgt, n_steps=n_steps, cond_mode=cond_mode)
    print(df_fp.to_string(index=False))
    corr_with_alpha(df_fp,
        ["sim_fp_source", "sim_fp_target", "fp_on_bits"],
        label="FP decoder")

    # Stochastic sampling
    print("\n[Stochastic (sample)]")
    rows = []
    for alpha in np.linspace(0, 1, n_steps):
        z    = (1 - alpha) * z_src + alpha * z_tgt
        cond = get_interp_cond(alpha, cond_src, cond_tgt, cond_mode)
        with torch.no_grad():
            for j in range(n_per_step):
                ids = model.smi_decoder.sample(
                    z, cond,
                    max_len=INTERP_MAX_LEN,
                    min_len=INTERP_MIN_LEN,
                    **INTERP_SAMPLE_KWARGS,
                )
                smi = decode_tokens(ids)
                p   = mol_props(smi) if check_valid(smi) else None
                if p is None:
                    continue
                rows.append({
                    "alpha":       alpha,
                    "sample_id":   j + 1,
                    **p,
                    "alkyl_proxy": get_long_alkyl_chain(smi, min_len=8),
                    "sim_source":  fp_tanimoto(smi, smi_src) if smi_src else np.nan,
                    "sim_target":  fp_tanimoto(smi, smi_tgt) if smi_tgt else np.nan,
                    "n_tokens":    len(ids),
                    "smiles":      smi,
                })

    df_raw = pd.DataFrame(rows)
    if df_raw.empty:
        print("  No valid samples.")
        return df_fp, df_raw, pd.DataFrame()

    df_summary = df_raw.groupby("alpha").agg({
        "mw":          "median",
        "logp":        "median",
        "tpsa":        "median",
        "amide":       "median",
        "alkyl_proxy": "median",
        "rot_bonds":   "median",
        "n_tokens":    "median",
    }).reset_index()

    print(df_summary.to_string(index=False))

    props = ["mw", "tpsa", "amide", "alkyl_proxy", "n_tokens"]
    corr_with_alpha(df_summary, props, label="sample median")

    return df_fp, df_raw, df_summary


# ── 실행 ─────────────────────────────────────────────────
pairs_custom = [
    ("Exenatide",    centroid_tuple, "Exenatide → Cluster centroid",   "source"),
    (centroid_tuple, "Semaglutide",  "Cluster centroid → Semaglutide", "source"),
    ("Semaglutide",  "Tirzepatide",  "Semaglutide → Tirzepatide",      "interp"),
    ("Tirzepatide",  "Retatrutide",  "Tirzepatide → Retatrutide",      "interp"),
]

segment_results = {}
for src, tgt, label, cond_mode in pairs_custom:
    df_fp, df_raw, df_sum = run_interp_custom(
        src, tgt, label,
        cond_mode=cond_mode,
        n_steps=5,
        n_per_step=30,
    )
    segment_results[label] = {"fp": df_fp, "raw": df_raw, "summary": df_sum}